# Manual feature ablation


Compare Activity, Diet and Bio feature groups in jointly fitted Ridge models, with shallow GBT and additional cohorts as supplementary checks. Core inputs must satisfy training-scope coverage and prediction-time availability. Single-group and leave-one-group-out contrasts separate group effects from addition order.


In [1]:
# --- Import dependencies ---

import json
import re
import time
import os

# Fix the physical-core count so joblib/loky does not emit a non-actionable Windows detection warning.
os.environ.setdefault("LOKY_MAX_CPU_COUNT", str(os.cpu_count() or 1))

import joblib
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from src.evaluation import classify_effect, evaluate, subject_metrics
from src.paths import DATA_ROOT, PROJECT_ROOT, validate_data_root

# --- Validate data access and configure artifact directories ---


validate_data_root()
ALIGN_DIR = PROJECT_ROOT / "results" / "aligned_data"
EDA_DIR = PROJECT_ROOT / "results" / "forecast_windows"
BASELINE_DIR = PROJECT_ROOT / "results" / "baselines"
ABLATION_DIR = PROJECT_ROOT / "results" / "manual_ablation"
ABLATION_DIR.mkdir(parents=True, exist_ok=True)

# --- Configure compact tabular displays ---

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 20)
pd.set_option("display.max_colwidth", 70)

In [2]:
# Load the baseline task, keys, model specifications, and upstream artifact paths.
baseline_registry = json.loads((BASELINE_DIR / "baseline_registry.json").read_text(encoding="utf-8"))
upstream_paths = baseline_registry["upstream_paths"]
WINDOW_KEY = baseline_registry["sample_keys"]["window_key"]
FOLD_KEY = baseline_registry["sample_keys"]["fold_key"]
PROTOCOL_A, PROTOCOL_B = baseline_registry["protocol_names"]["A"], baseline_registry["protocol_names"]["B"]

# data alignment availability registry; window construction training-scope candidate registry, training scope and glucose-history feature registry.
availability_registry = pd.read_csv(ALIGN_DIR / "availability_registry.csv")
candidate_factors = pd.read_csv(EDA_DIR / "candidate_factors.csv")
training_scope = pd.read_csv(EDA_DIR / "training_eda_scope.csv", parse_dates=["scope_start", "scope_end"])
glucose_registry = pd.read_csv(EDA_DIR / "feature_registry.csv")
all_folds = pd.read_parquet(PROJECT_ROOT / upstream_paths["fold_assignments"], columns=FOLD_KEY + ["split", "admitted_to_split"])

In [3]:
# Training-scope coverage (Protocol B training subjects within their Protocol A training period).
scope = training_scope[["subject", "scope_start", "scope_end"]]

def subjects_with_values(frame: pd.DataFrame, fields: list, time_column: str = None) -> dict:
    frame = frame.merge(scope, on="subject", how="inner")
    if time_column is not None:
        frame = frame.loc[frame[time_column].between(frame["scope_start"], frame["scope_end"])]
    return {field: int(frame.loc[frame[field].notna(), "subject"].nunique()) for field in fields}


activity_fields = availability_registry.loc[availability_registry["source_object"].eq("activity_canonical"), "source_field"].tolist()
meal_fields = availability_registry.loc[availability_registry["source_object"].eq("food_entries"), "source_field"].unique().tolist()
meal_fields = [f for f in meal_fields if f != "image_path"]
bio_fields = availability_registry.loc[availability_registry["feature_group"].isin(["bio_baseline", "bio_lab_or_fingerstick"]), "source_field"].tolist()

coverage = {}
coverage.update(subjects_with_values(
    pd.read_parquet(ALIGN_DIR / "activity_canonical.parquet", columns=["subject", "timestamp"] + activity_fields),
    activity_fields, "timestamp"))
coverage.update(subjects_with_values(
    pd.read_parquet(ALIGN_DIR / "meal_events.parquet", columns=["subject", "timestamp"] + meal_fields),
    meal_fields, "timestamp"))
coverage.update(subjects_with_values(pd.read_parquet(ALIGN_DIR / "bio_canonical.parquet", columns=["subject"] + bio_fields), bio_fields))
static = pd.read_parquet(ALIGN_DIR / "static_canonical.parquet")
gut_columns = [c for c in static.columns if c.startswith("gut__")]
microbiome_columns = [c for c in static.columns if c.startswith("microbiome__")]
static_scoped = static.merge(scope[["subject"]], on="subject")
coverage["gut_health_scores"] = int(static_scoped[gut_columns].notna().any(axis=1).sum())
coverage["microbiome_indicators"] = int(static_scoped[microbiome_columns].notna().any(axis=1).sum())
n_training_subjects = scope["subject"].nunique()

display(pd.Series(coverage, name="training_subjects_with_data").to_frame())

,training_subjects_with_data
heart_rate_bpm,27
activity_calories_per_minute,27
activity_mets_stored_x10,18
activity_met,18
activity_intensity_raw,9
steps_raw,0
meal_type,27
meal_calories,27
carbs,27
protein,27


Coverage uses the shared TRAIN scope defined in `03_eda_task_samples.ipynb`.


In [4]:
# Factor-group assignment of every availability-registry row. Pre-selection follows the training-scope EDA candidate registry;
# eligibility follows the availability policy. Order of rules matters: the first matching rule applies.
GROUP_OF_REGISTRY_GROUP = {
    "glucose_history": "G", "activity_minute": "A", "meal_log": "D", "meal_completion": "D", "meal_photo": "D",
    "bio_baseline": "B", "bio_lab_or_fingerstick": "B", "gut_health_static": "Gut", "microbiome_static": "Microbiome",
    "temporal_context": "none", "identifier_or_source_metadata": "none", "alignment_audit": "none",
    "derived_feature_policy": "policy",
}
GROUP_NAMES = {"G": "Glucose History", "A": "Activity", "D": "Diet", "B": "Bio", "Gut": "Gut",
               "Microbiome": "Microbiome", "none": "not a factor group", "policy": "pipeline policy"}
EXCLUDED_SCOPES = {"excluded_from_model_features", "offline_audit_only", "excluded_from_causal_features"}
eda_status = dict(zip(candidate_factors["candidate"].str.replace("bio__", "", regex=False), candidate_factors["eda_status"]))
DUPLICATE_OF = {"activity_mets_stored_x10": "activity_met"}
UNIT_UNRESOLVED = {"hba1c_lab_raw": "HbA1c unit conflict (percent vs mmol/mol) unresolved upstream"}


def assign(row: pd.Series) -> tuple:
    """Return (factor_status, rationale)."""
    group, field, scope_ = GROUP_OF_REGISTRY_GROUP[row["feature_group"]], row["source_field"], row["model_input_scope"]
    status = eda_status.get(field)
    if group == "policy":
        return "applies_to_all_groups", "pipeline rule from the availability registry"
    if scope_ in EXCLUDED_SCOPES:
        return "excluded_upstream", f"availability registry scope: {scope_}"
    if group == "none":
        return "not_assigned_to_factor_group", f"not part of G/A/D/B; availability registry scope: {scope_}"
    if group == "Microbiome":
        return "deferred_to_optional_studies", "high-dimensional subject-level input; deferred to the optional studies"
    if group == "Gut":
        return "optional_supplement_not_core", f"training-EDA status: {status}; small optional supplement"
    if field in DUPLICATE_OF:
        return "excluded_duplicate_representation", f"same signal as {DUPLICATE_OF[field]}"
    if row["safe_for_prediction_at_t"].startswith("no_"):
        return "excluded_not_available_at_t", f"availability registry: {row['safe_for_prediction_at_t']}"
    if status is None:
        return "not_preselected", "not a training-EDA candidate"
    if status == "candidate_sensitivity_device":
        return "sensitivity_device_not_core", "Dexcom is a separate target/device task"
    if status.startswith("not_recommended") or status.startswith("excluded"):
        return "excluded_by_training_eda", f"training-EDA status: {status}"
    if status == "candidate_offline_analysis_only":
        extra = f"; {UNIT_UNRESOLVED[field]}" if field in UNIT_UNRESOLVED else ""
        return ("offline_sensitivity_only" + ("_pending_unit_decision" if field in UNIT_UNRESOLVED else ""),
                "availability registry: result timing unverified; offline assumption required" + extra)
    if status == "candidate_with_availability_assumption":
        return "core_with_availability_assumption", "availability registry: usable only after record creation; timing assumed"
    if group == "G":
        return "core_via_glucose_history_features", "represented by the glucose-history feature registry"
    return "core", f"training-EDA status: {status}"


# Registry rows: Gut and Microbiome are aggregated to one row each (field counts kept); all other rows kept individually.
rows = []
for registry_group, part in availability_registry.groupby("feature_group", sort=False):
    if registry_group in ("gut_health_static", "microbiome_static"):
        field = "gut_health_scores" if registry_group == "gut_health_static" else "microbiome_indicators"
        first = part.iloc[0].copy()
        first["source_field"], first["candidate_feature"] = field, field
        part = pd.DataFrame([first]).assign(
            registry_ids=part["registry_id"].iloc[0] if len(part) == 1 else f"{part['registry_id'].iloc[0]}..{part['registry_id'].iloc[-1]}",
            n_fields=int(part["n_fields"].sum()))
    else:
        part = part.assign(registry_ids=part["registry_id"], n_fields=1)
    rows.append(part)
registry = pd.concat(rows, ignore_index=True)
registry[["factor_status", "rationale"]] = registry.apply(assign, axis=1, result_type="expand")
registry["group_code"] = registry["feature_group"].map(GROUP_OF_REGISTRY_GROUP)
registry["group_name"] = registry["group_code"].map(GROUP_NAMES)
registry["training_eda_status"] = registry["source_field"].map(eda_status)
registry["training_subjects_with_data"] = registry["source_field"].map(coverage)
registry["record_type"] = "availability_field"

# Glucose History representation: the glucose-history feature registry (predictors allowed by the registry).
g_features = glucose_registry.loc[glucose_registry["allowed_as_predictor"]]
g_rows = pd.DataFrame({
    "record_type": "glucose_history_feature", "group_code": "G", "group_name": GROUP_NAMES["G"],
    "candidate_feature": g_features["feature_name"], "source_object": "glucose_history_samples",
    "source_field": g_features["source_signal"], "registry_ids": "glucose-history feature registry", "n_fields": 1,
    "observation_level": "sample_window",
    "safe_for_prediction_at_t": "conditional_strictly_historical", "model_input_scope": "candidate_with_causal_cutoff",
    "factor_status": "core", "rationale": "glucose-history predictor from the feature registry",
    "training_eda_status": "candidate_for_ablation",
})
columns = ["record_type", "group_code", "group_name", "candidate_feature", "source_object", "source_field",
           "registry_ids", "n_fields", "observation_level", "safe_for_prediction_at_t", "model_input_scope",
           "offline_analysis_only", "training_eda_status", "training_subjects_with_data", "factor_status", "rationale"]
feature_group_registry = pd.concat([registry, g_rows], ignore_index=True).reindex(columns=columns)

# Completeness: every availability-registry row is represented exactly once; each core factor group has at least one core field.
covered_ids = set(availability_registry.loc[~availability_registry["feature_group"].isin(["gut_health_static", "microbiome_static"]), "registry_id"])
assert registry["n_fields"].sum() == availability_registry["n_fields"].sum()
assert not feature_group_registry.duplicated(["record_type", "group_code", "candidate_feature"]).any()
core = feature_group_registry["factor_status"].str.startswith("core")
assert set(feature_group_registry.loc[core, "group_code"]) == {"G", "A", "D", "B"}

display(feature_group_registry.loc[feature_group_registry["group_code"].isin(["A", "D", "B"]),
                                   ["group_code", "candidate_feature", "factor_status", "training_subjects_with_data", "rationale"]])

,group_code,candidate_feature,factor_status,training_subjects_with_data,rationale
2,A,heart_rate_bpm,core,27.0,training-EDA status: candidate_for_ablation
3,A,activity_calories_per_minute,core,27.0,training-EDA status: candidate_for_ablation
4,A,activity_mets_stored_x10,excluded_duplicate_representation,18.0,same signal as activity_met
5,A,activity_met,core,18.0,training-EDA status: candidate_for_ablation
6,A,activity_intensity_raw,excluded_by_training_eda,9.0,training-EDA status: not_recommended_limited_coverage
7,A,steps_raw,excluded_by_training_eda,0.0,training-EDA status: not_recommended_limited_coverage
8,D,meal_type,core_with_availability_assumption,27.0,availability registry: usable only after record creation; timing a...
9,D,meal_calories,core_with_availability_assumption,27.0,availability registry: usable only after record creation; timing a...
10,D,carbs,core_with_availability_assumption,27.0,availability registry: usable only after record creation; timing a...
11,D,protein,core_with_availability_assumption,27.0,availability registry: usable only after record creation; timing a...


MET coverage varies by participant schema, so activity features include availability indicators. Diet inputs assume availability at log creation; a sensitivity analysis delays availability by 30 minutes.


In [5]:
# Core ablation configurations: unique feature-group sets with every role they serve.
FULL = ("G", "A", "D", "B")
roles = {}

def add(groups: tuple, role: str):
    roles.setdefault(tuple(g for g in FULL if g in groups), []).append(role)


add(("G",), "glucose-only reference")
for k in range(2, len(FULL) + 1):
    add(FULL[:k], f"incremental step {k - 1}")
for g in FULL[1:]:
    add(("G", g), "G + single group")
for g in FULL[1:]:
    add(tuple(x for x in FULL if x != g), f"full model without {g}")
add(FULL, "full model")

configurations = pd.DataFrame([
{"record_type": "configuration", "config_id": "+".join(groups), "feature_groups": "+".join(groups),
     **{f"includes_{g}": g in groups for g in FULL}, "roles": "; ".join(r), "core": True,
     "protocols": "Within-subject Future Prediction; Unseen-subject Generalization"}
    for groups, r in roles.items()
])
config_of = dict(zip(configurations["feature_groups"], configurations["config_id"]))

# Planned comparisons; order effects are separated from single-group and leave-one-group-out evidence.
contrasts = []
for k in range(1, len(FULL)):
    contrasts.append(("incremental", "+".join(FULL[:k]), "+".join(FULL[:k + 1]), FULL[k]))
for g in FULL[1:]:
    contrasts.append(("single_group_addition", "G", f"G+{g}", g))
for g in FULL[1:]:
    contrasts.append(("leave_one_group_out", "+".join(x for x in FULL if x != g), "+".join(FULL), g))
# A contrast serving several roles (e.g. G -> G+A is both incremental step 1 and a single-group addition) is kept once.
merged = {}
for t, a, b, g in contrasts:
    merged.setdefault((a, b, g), []).append(t)
contrast_table = pd.DataFrame([
{"record_type": "contrast", "config_id": f"{a}->{b}", "contrast_type": "; ".join(types), "from_config": config_of[a],
     "to_config": config_of[b], "feature_groups": f"{a} -> {b}", "group_tested": g, "core": True}
    for (a, b, g), types in merged.items()
])

# Planned non-core sensitivity variants; their inputs are specified before any fitting.
sensitivity = pd.DataFrame([
    {"record_type": "sensitivity", "config_id": "diet_delay", "feature_groups": "G+A+D(delayed availability)+B",
     "roles": "diet availability-delay sensitivity (delay rule specified with the Activity and Diet features)", "core": False},
    {"record_type": "sensitivity", "config_id": "offline_lab_bio", "feature_groups": "G+A+D+B+B_lab(offline)",
     "roles": "offline laboratory Bio sensitivity (fields and HbA1c handling specified with the Bio features)", "core": False},
    {"record_type": "sensitivity", "config_id": "gut", "feature_groups": "G+A+D+B+Gut",
     "roles": "optional Gut supplement on the Gut-available common cohort", "core": False},
])

ablation_matrix = pd.concat([configurations, contrast_table, sensitivity], ignore_index=True)

# Coverage of the required design.
required = {"G", "G+A", "G+A+D", "G+A+D+B", "G+D", "G+B", "G+D+B", "G+A+B"}
assert set(configurations["feature_groups"]) == required and configurations["feature_groups"].is_unique
assert len(contrasts) == 9 and not contrast_table.duplicated(["from_config", "to_config"]).any()

display(configurations[["config_id", "feature_groups", "roles"]])
display(contrast_table[["config_id", "contrast_type", "feature_groups", "group_tested"]])
display(sensitivity[["config_id", "feature_groups", "roles"]])

,config_id,feature_groups,roles
0,G,G,glucose-only reference
1,G+A,G+A,incremental step 1; G + single group
2,G+A+D,G+A+D,incremental step 2; full model without B
3,G+A+D+B,G+A+D+B,incremental step 3; full model
4,G+D,G+D,G + single group
5,G+B,G+B,G + single group
6,G+D+B,G+D+B,full model without A
7,G+A+B,G+A+B,full model without D


,config_id,contrast_type,feature_groups,group_tested
0,G->G+A,incremental; single_group_addition,G -> G+A,A
1,G+A->G+A+D,incremental,G+A -> G+A+D,D
2,G+A+D->G+A+D+B,incremental; leave_one_group_out,G+A+D -> G+A+D+B,B
3,G->G+D,single_group_addition,G -> G+D,D
4,G->G+B,single_group_addition,G -> G+B,B
5,G+D+B->G+A+D+B,leave_one_group_out,G+D+B -> G+A+D+B,A
6,G+A+B->G+A+D+B,leave_one_group_out,G+A+B -> G+A+D+B,D


,config_id,feature_groups,roles
0,diet_delay,G+A+D(delayed availability)+B,diet availability-delay sensitivity (delay rule specified with the...
1,offline_lab_bio,G+A+D+B+B_lab(offline),offline laboratory Bio sensitivity (fields and HbA1c handling spec...
2,gut,G+A+D+B+Gut,optional Gut supplement on the Gut-available common cohort


Nine designed contrasts reduce to seven unique contrasts because two serve two roles each. The model family and common cohorts are fixed in the common-cohort model comparison section; the Sparacino AR(1)-WLS baseline is univariate and cannot take exogenous groups.

In [6]:
# Persist the feature-group registry and ablation matrix.
feature_group_registry.to_csv(ABLATION_DIR / "feature_group_registry.csv", index=False, encoding="utf-8", lineterminator="\n")
ablation_matrix.to_csv(ABLATION_DIR / "ablation_matrix.csv", index=False, encoding="utf-8", lineterminator="\n")

`feature_group_registry.csv` assigns feature groups and availability. `ablation_matrix.csv` defines eight core configurations, seven unique contrasts and three sensitivity variants.


## Activity and diet feature engineering

All lookbacks lie within \([t-120,t]\). Unobserved activity is not rest or zero, and unknown intake remains NaN. The `diet_delay` sensitivity shifts Diet availability to 30 minutes after log time.


In [7]:
# Core fields of the Activity and Diet groups, from the feature-group registry built above.
core_fields = feature_group_registry.loc[feature_group_registry["factor_status"].str.startswith("core")]
ACTIVITY_FIELDS = core_fields.loc[core_fields["group_code"].eq("A"), "source_field"].tolist()
DIET_FIELDS = core_fields.loc[core_fields["group_code"].eq("D"), "source_field"].tolist()
NUTRIENTS = [f for f in DIET_FIELDS if f != "meal_type"]

# Task and sample keys (window construction artifacts named in the baseline registry).
task = json.loads((PROJECT_ROOT / upstream_paths["task_protocol"]).read_text(encoding="utf-8"))["primary_task"]
HISTORY_MIN = task["history_minutes"]
samples = pd.read_parquet(PROJECT_ROOT / upstream_paths["glucose_history_samples"])
assert (samples["origin_time"] - samples["history_start_time"]).eq(pd.Timedelta(minutes=HISTORY_MIN)).all()

# Planned lookbacks: all inside the history [t - HISTORY_MIN, t]; diet availability delay of the diet_delay sensitivity.
WINDOWS = [HISTORY_MIN // 4, HISTORY_MIN]
DIET_DELAY_MIN = 30
assert max(WINDOWS) <= HISTORY_MIN and DIET_DELAY_MIN <= min(WINDOWS)

In [8]:
# Trailing activity summaries on the contiguous 1-minute grid; only published, observed minutes contribute.
activity = pd.read_parquet(ALIGN_DIR / "activity_canonical.parquet",
                           columns=["subject", "timestamp", "is_published_source_row"] + ACTIVITY_FIELDS)
activity = activity.sort_values(["subject", "timestamp"]).reset_index(drop=True)
assert activity.groupby("subject")["timestamp"].diff().dropna().eq(pd.Timedelta(minutes=1)).all()
for field in ACTIVITY_FIELDS:
    activity.loc[~activity["is_published_source_row"], field] = np.nan
subject_start = activity.groupby("subject")["timestamp"].first()
origin_pos = ((samples["origin_time"] - samples["subject"].map(subject_start)) // pd.Timedelta(minutes=1)).to_numpy()
assert (origin_pos - HISTORY_MIN >= 0).all()

SHORT = {"heart_rate_bpm": "hr", "activity_calories_per_minute": "calories", "activity_met": "met"}
subject_arrays = {s: {f: part[f].to_numpy(dtype=float) for f in ACTIVITY_FIELDS} for s, part in activity.groupby("subject")}
activity_features = {}
for field in ACTIVITY_FIELDS:
    short = SHORT[field]
    for window in WINDOWS:
        columns = {k: np.full(len(samples), np.nan) for k in ["mean", "max", "sum", "coverage"]}
        for subject, idx in samples.groupby("subject").indices.items():
            values = subject_arrays[subject][field]
            observed = ~np.isnan(values)
            csum_v = np.r_[0.0, np.cumsum(np.where(observed, values, 0.0))]
            csum_n = np.r_[0, np.cumsum(observed)]
            rolling_max = pd.Series(values).rolling(window + 1, min_periods=1).max().to_numpy()
            hi, lo = origin_pos[idx] + 1, origin_pos[idx] - window
            n = csum_n[hi] - csum_n[lo]
            s = csum_v[hi] - csum_v[lo]
            with np.errstate(invalid="ignore", divide="ignore"):
                columns["mean"][idx] = np.where(n > 0, s / n, np.nan)
            columns["sum"][idx] = np.where(n > 0, s, np.nan)
            columns["max"][idx] = np.where(n > 0, rolling_max[origin_pos[idx]], np.nan)
            columns["coverage"][idx] = n / (window + 1)
        activity_features[f"act_{short}_mean_{window}m"] = columns["mean"]
        activity_features[f"act_{short}_max_{window}m"] = columns["max"]
        if field == "activity_calories_per_minute":
            activity_features[f"act_{short}_sum_observed_{window}m"] = columns["sum"]
        activity_features[f"act_{short}_coverage_{window}m"] = columns["coverage"]
        activity_features[f"act_{short}_missing_{window}m"] = (columns["coverage"] == 0).astype("int8")
activity_features = pd.DataFrame(activity_features)

In [9]:
# Logged meal events (meal anchors; photo-only entries, Amount Consumed and episode metadata are not used).
meals = pd.read_parquet(ALIGN_DIR / "meal_events.parquet", columns=["subject", "timestamp", "meal_event_id", "meal_type_normalized"] + NUTRIENTS)
assert not meals.duplicated(["subject", "timestamp"]).any() and meals["timestamp"].dt.second.eq(0).all()
MEAL_TYPES = sorted(meals["meal_type_normalized"].dropna().unique())
subject_end = activity.groupby("subject")["timestamp"].last()


def diet_block(delay: int) -> pd.DataFrame:
    """Meals count when their availability time (log time + delay) <= t and log time >= t - window."""
    suffix = "" if delay == 0 else f"__avail_delay{delay}"
    out = {}
    last_minutes = np.full(len(samples), np.nan)
    last_values = {k: np.full(len(samples), np.nan) for k in ["carbs", "meal_calories"] + MEAL_TYPES}
    window_cols = {(w, k): np.full(len(samples), np.nan) for w in WINDOWS for k in ["count"] + NUTRIENTS}
    for subject, idx in samples.groupby("subject").indices.items():
        length = int((subject_end[subject] - subject_start[subject]) // pd.Timedelta(minutes=1)) + 1
        part = meals.loc[meals["subject"].eq(subject)]
        pos = ((part["timestamp"] - subject_start[subject]) // pd.Timedelta(minutes=1)).to_numpy()
        event = np.zeros(length)
        event[pos] = 1
        csum_event = np.r_[0.0, np.cumsum(event)]
        # Latest logged event at or before each grid position.
        last_pos = np.maximum.accumulate(np.where(event > 0, np.arange(length), -1))
        o = origin_pos[idx]
        avail_hi = o - delay  # latest admissible log position
        for window in WINDOWS:
            lo = o - window
            n = csum_event[avail_hi + 1] - csum_event[lo]
            window_cols[(window, "count")][idx] = n
            for nutrient in NUTRIENTS:
                values = np.zeros(length)
                nonnull = np.zeros(length)
                values[pos] = part[nutrient].fillna(0).to_numpy()
                nonnull[pos] = part[nutrient].notna().to_numpy()
                s = np.r_[0.0, np.cumsum(values)][avail_hi + 1] - np.r_[0.0, np.cumsum(values)][lo]
                k = np.r_[0.0, np.cumsum(nonnull)][avail_hi + 1] - np.r_[0.0, np.cumsum(nonnull)][lo]
                # Unknown when no meal is logged in the window or any logged meal lacks this nutrient.
                window_cols[(window, nutrient)][idx] = np.where((n > 0) & (k == n), s, np.nan)
        lp = last_pos[avail_hi]
        valid = lp >= o - HISTORY_MIN
        last_minutes[idx] = np.where(valid, o - lp, np.nan)
        # Attributes of the latest admissible meal, looked up through a grid-position -> event-row map.
        row_at = np.full(length, -1)
        row_at[pos] = np.arange(len(part))
        rows = row_at[np.where(valid, lp, 0)]
        for key in ["carbs", "meal_calories"]:
            last_values[key][idx] = np.where(valid, part[key].to_numpy()[rows], np.nan)
        for meal_type in MEAL_TYPES:
            last_values[meal_type][idx] = np.where(valid, (part["meal_type_normalized"].to_numpy()[rows] == meal_type).astype(float), np.nan)
    for window in WINDOWS:
        out[f"diet_logged_meal_count_{window}m{suffix}"] = window_cols[(window, "count")]
        out[f"diet_no_logged_meal_{window}m{suffix}"] = (window_cols[(window, "count")] == 0).astype("int8")
        for nutrient in NUTRIENTS:
            out[f"diet_logged_{nutrient}_sum_{window}m{suffix}"] = window_cols[(window, nutrient)]
    out[f"diet_minutes_since_last_logged_meal{suffix}"] = last_minutes
    out[f"diet_last_meal_carbs{suffix}"] = last_values["carbs"]
    out[f"diet_last_meal_calories{suffix}"] = last_values["meal_calories"]
    for meal_type in MEAL_TYPES:
        out[f"diet_last_meal_type_{meal_type.lower()}{suffix}"] = last_values[meal_type]
    return pd.DataFrame(out)

diet_features = pd.concat([diet_block(0), diet_block(DIET_DELAY_MIN)], axis=1)

In [10]:
# One row per window, keyed exactly as in window construction.
features = pd.concat([samples[WINDOW_KEY].reset_index(drop=True), activity_features, diet_features], axis=1)

def meta(name: str) -> dict:
    delayed = name.endswith(f"__avail_delay{DIET_DELAY_MIN}")
    base = name.replace(f"__avail_delay{DIET_DELAY_MIN}", "")
    window = next((w for w in WINDOWS if base.endswith(f"_{w}m")), HISTORY_MIN)
    if base.startswith("act_"):
        field = {v: k for k, v in SHORT.items()}[base.split("_")[1]]
        return {"feature_name": name, "group_code": "A", "variant": "primary", "source_object": "activity_canonical",
                "source_field": field, "lookback_interval": f"[t-{window}min, t]"}
    field = next((n for n in NUTRIENTS if f"_{n}_" in base or base.endswith(f"_{n}")), "meal_type" if "type" in base else "meal_event")
    upper = f"t-{DIET_DELAY_MIN}min" if delayed else "t"
    return {"feature_name": name, "group_code": "D", "variant": "delayed availability" if delayed else "primary",
            "source_object": "meal_events", "source_field": field,
            "lookback_interval": f"log time in [t-{window}min, {upper}]"}
activity_diet_feature_registry = pd.DataFrame([meta(c) for c in features.columns if c not in WINDOW_KEY])
assert set(activity_diet_feature_registry["source_field"]) - {"meal_event"} <= set(ACTIVITY_FIELDS) | set(DIET_FIELDS)
display(activity_diet_feature_registry.groupby(["group_code", "variant"]).size().rename("features").to_frame())

features
group_code variant                       
A          primary                     26
D          delayed availability        21
           primary                     21

In [11]:
# --- Validate activity and diet feature construction ---

audit = {}
audit["no label, target-time or split columns"] = not any(c in features.columns for c in ["target_value", "target_time", "split", "evaluation_protocol"])

# --- Independently recompute a fixed sample of Protocol B training windows ---

train_keys = all_folds.loc[all_folds["admitted_to_split"] & all_folds["split"].eq("train")
                          & all_folds["evaluation_protocol"].eq(PROTOCOL_B), WINDOW_KEY]
check = features.merge(train_keys, on=WINDOW_KEY).sample(300, random_state=0)
activity_by_subject = {s: part.set_index("timestamp") for s, part in activity.groupby("subject")}
meals_by_subject = {s: part.set_index("timestamp") for s, part in meals.groupby("subject")}
max_diff, max_source_offset = 0.0, -np.inf
for row in check.itertuples(index=False):
    t = row.origin_time
    for field in ACTIVITY_FIELDS:
        for window in WINDOWS:
            seg = activity_by_subject[row.subject].loc[t - pd.Timedelta(minutes=window): t, field].dropna()
            expected = seg.mean() if len(seg) else np.nan
            got = getattr(row, f"act_{SHORT[field]}_mean_{window}m")
            max_diff = max(max_diff, 0.0 if (np.isnan(expected) and np.isnan(got)) else abs(expected - got))
            if len(seg):
                max_source_offset = max(max_source_offset, (seg.index.max() - t) / pd.Timedelta(minutes=1))
    for delay, suffix in [(0, ""), (DIET_DELAY_MIN, f"__avail_delay{DIET_DELAY_MIN}")]:
        subject_meals = meals_by_subject[row.subject] if row.subject in meals_by_subject else meals.iloc[0:0].set_index("timestamp")
        for window in WINDOWS:
            seg = subject_meals.loc[t - pd.Timedelta(minutes=window): t - pd.Timedelta(minutes=delay)]
            expected = seg["carbs"].sum() if len(seg) and seg["carbs"].notna().all() else np.nan
            got = getattr(row, f"diet_logged_carbs_sum_{window}m{suffix}")
            max_diff = max(max_diff, 0.0 if (np.isnan(expected) and np.isnan(got)) else abs(expected - got))
            if len(seg):
                max_source_offset = max(max_source_offset, (seg.index.max() + pd.Timedelta(minutes=delay) - t) / pd.Timedelta(minutes=1))
audit["independent recomputation matches (300 training windows)"] = max_diff < 1e-9
audit["no contributing source or availability time after t"] = max_source_offset <= 0

# --- Confirm delayed availability and missing-value behavior ---

for window in WINDOWS:
    primary, delayed = features[f"diet_logged_meal_count_{window}m"], features[f"diet_logged_meal_count_{window}m__avail_delay{DIET_DELAY_MIN}"]
    audit.setdefault("delayed diet counts <= primary counts", True)
    audit["delayed diet counts <= primary counts"] &= bool((delayed <= primary).all())
# Unknown is never encoded as zero intake: nutrient sums are NaN exactly when no admissible meal exists.
for window in WINDOWS:
    no_meal = features[f"diet_no_logged_meal_{window}m"].eq(1)
    audit.setdefault("no logged meal -> nutrient sums NaN (not 0)", True)
    audit["no logged meal -> nutrient sums NaN (not 0)"] &= bool(features.loc[no_meal, f"diet_logged_carbs_sum_{window}m"].isna().all())
    missing = features[f"act_hr_missing_{window}m"].eq(1)
    audit.setdefault("activity missing -> summaries NaN (not 0)", True)
    audit["activity missing -> summaries NaN (not 0)"] &= bool(features.loc[missing, f"act_hr_mean_{window}m"].isna().all())

audit_table = pd.Series(audit, name="passed").to_frame()
assert audit_table["passed"].all(), audit_table
print(f"Recomputation max abs difference: {max_diff:.2e}; latest contributing time relative to t: {max_source_offset:.0f} min")

# --- Summarize training-window missingness ---

train_all = all_folds.loc[all_folds["admitted_to_split"] & all_folds["split"].eq("train")]
missing_summary = (train_all.merge(features, on=WINDOW_KEY).groupby("evaluation_protocol")
                   [[c for c in features.columns if c.endswith("_missing_120m") or c.startswith("diet_no_logged_meal_")]].mean().T)
display(missing_summary.round(3))

Recomputation max abs difference: 1.91e-12; latest contributing time relative to t: 0 min


evaluation_protocol,Unseen-subject Generalization,Within-subject Future Prediction
act_hr_missing_120m,0.081,0.031
act_calories_missing_120m,0.033,0.001
act_met_missing_120m,0.346,0.239
diet_no_logged_meal_30m,0.925,0.920
diet_no_logged_meal_120m,0.725,0.706
diet_no_logged_meal_30m__avail_delay30,0.998,0.998
diet_no_logged_meal_120m__avail_delay30,0.789,0.776


Unknown intake is kept as NaN with explicit flags rather than imputed, because most training windows contain no logged meal within 120 minutes and MET missingness is largely structural (subjects without the MET schema).

Meals logged after \(t\) never enter the features: the independent recomputation above counts only meals whose log time (plus the delay in the diet-delay variant) is at or before \(t\).


In [12]:
# Persist activity and diet features with their registry.
activity_diet_features_path = ABLATION_DIR / "activity_diet_features.parquet"
activity_diet_registry_path = ABLATION_DIR / "activity_diet_feature_registry.csv"
features.to_parquet(activity_diet_features_path, index=False)
activity_diet_feature_registry.to_csv(activity_diet_registry_path, index=False, encoding="utf-8", lineterminator="\n")

## Bio and optional gut features

Core Bio features are age, gender and BMI. Fasting glucose, fasting insulin, triglycerides and HDL form the offline laboratory sensitivity set (`offline_lab_bio`) because their result times are unverified. HbA1c is not produced while its unit remains unconfirmed. The 22 Gut scores (optional supplement `gut`) are encoded both as ordinal values and as one indicator per level.

In [13]:
# Decisions for Bio and Gut, from the feature-group registry built above.
bio_rows = feature_group_registry.loc[feature_group_registry["group_code"].eq("B") & feature_group_registry["record_type"].eq("availability_field")]
BIO_CORE = bio_rows.loc[bio_rows["factor_status"].eq("core"), "candidate_feature"].tolist()
BIO_SENSITIVITY = bio_rows.loc[bio_rows["factor_status"].eq("offline_sensitivity_only"), "candidate_feature"].tolist()
BIO_PENDING = bio_rows.loc[bio_rows["factor_status"].str.contains("pending_unit_decision"), "candidate_feature"].tolist()
gut_row = feature_group_registry.loc[feature_group_registry["group_code"].eq("Gut")].iloc[0]
assert gut_row["factor_status"] == "optional_supplement_not_core"

# Static canonical table (read above for the coverage counts) and its Gut columns.
GUT_COLUMNS = [c for c in static.columns if c.startswith("gut__")]
assert len(GUT_COLUMNS) == int(gut_row["n_fields"])

# Gut level definitions from the formal data dictionary.
gut_dictionary = pd.read_csv(DATA_ROOT / "DataDictionary_Gut_Health_Test.csv")
level_maps = {field: dict((int(k), v.strip()) for k, v in re.findall(r"(\d+)\s*=\s*([A-Za-z ]+?)(?:,|$)", value))
              for field, value in zip(gut_dictionary["Field"], gut_dictionary["Value"])}
assert {f"gut__{f}" for f in level_maps} == set(GUT_COLUMNS)

# Sample keys and protocol assignment (window construction artifacts named in the baseline registry).
windows = samples[WINDOW_KEY].copy()
admitted_folds = all_folds.loc[all_folds["admitted_to_split"]]
b_train_subjects = set(admitted_folds.loc[admitted_folds["evaluation_protocol"].eq(PROTOCOL_B) & admitted_folds["split"].eq("train"), "subject"])

print(f"Bio core: {BIO_CORE}")
print(f"Bio offline laboratory sensitivity: {BIO_SENSITIVITY}; pending unit decision: {BIO_PENDING}")
print(f"Gut scores: {len(GUT_COLUMNS)}; dictionary levels: {sorted(set(map(tuple, (sorted(m.items()) for m in level_maps.values()))))}")

Bio core: ['bio__age', 'bio__gender', 'bio__bmi']
Bio offline laboratory sensitivity: ['bio__fasting_glucose_lab_mg_dl', 'bio__fasting_insulin_raw', 'bio__triglycerides_mg_dl', 'bio__hdl_mg_dl']; pending unit decision: ['bio__hba1c_lab_raw']
Gut scores: 22; dictionary levels: [((1, 'Not Optimal'), (2, 'Average'), (3, 'Good'))]


In [14]:
# Bio features: raw numeric values; gender as indicators over categories seen in Protocol B training subjects.
bio = static[["subject"]].copy()
source_of = {}
for source in BIO_CORE + BIO_SENSITIVITY:
    values = static[source]
    prefix = "bio_" if source in BIO_CORE else "bio_lab_"
    if pd.api.types.is_numeric_dtype(values):
        column = prefix + source.replace("bio__", "")
        bio[column] = values.astype(float)
        source_of[column] = source
    else:
        categories = sorted(static.loc[static["subject"].isin(b_train_subjects), source].dropna().unique())
        for category in categories[1:]:  # first training category is the reference level
            column = f"{prefix}{source.replace('bio__', '')}_{str(category).lower()}"
            bio[column] = np.where(values.isin(categories), (values == category).astype(float), np.nan)
            source_of[column] = source
        gender_reference = categories[0]
bio["bio_values_available"] = static["bio_values_available"].astype("int8")
source_of["bio_values_available"] = "bio record"

display(bio.drop(columns="subject").notna().sum().rename("subjects_with_value").to_frame())

,subjects_with_value
bio_age,45
bio_gender_m,45
bio_bmi,45
bio_lab_fasting_glucose_lab_mg_dl,45
bio_lab_fasting_insulin_raw,45
bio_lab_triglycerides_mg_dl,45
bio_lab_hdl_mg_dl,45
bio_values_available,45


In [15]:
# Gut scores keep their three ordered levels: ordinal (1-3) and categorical (one indicator per level) encodings.
def slug(name: str) -> str:
    return re.sub(r"[^0-9a-z]+", "_", name.lower()).strip("_")

gut = static[["subject"]].copy()
for column in GUT_COLUMNS:
    field = column.replace("gut__", "")
    levels = level_maps[field]
    values = static[column]
    gut[f"gut_ord_{slug(field)}"] = values.astype(float)
    source_of[f"gut_ord_{slug(field)}"] = column
    for level, label in sorted(levels.items()):
        gut[f"gut_cat_{slug(field)}__{slug(label)}"] = np.where(values.notna(), (values == level).astype(float), np.nan)
        source_of[f"gut_cat_{slug(field)}__{slug(label)}"] = column
gut["gut_values_available"] = static["gut_values_available"].astype("int8")
source_of["gut_values_available"] = "gut record"
ordinal_columns = [c for c in gut.columns if c.startswith("gut_ord_")]
categorical_columns = [c for c in gut.columns if c.startswith("gut_cat_")]
# Each available subject has exactly one level indicator set per score.
row_sums = gut[categorical_columns].to_numpy().reshape(len(gut), len(GUT_COLUMNS), -1).sum(axis=2)
assert np.allclose(row_sums[gut["gut_values_available"].eq(1)], 1) and np.isnan(row_sums[gut["gut_values_available"].eq(0)]).all()
print(f"Gut: {len(ordinal_columns)} ordinal + {len(categorical_columns)} categorical columns; "
      f"subjects with Gut values: {int(gut['gut_values_available'].sum())} of {len(gut)}")

Gut: 22 ordinal + 66 categorical columns; subjects with Gut values: 42 of 45


In [16]:
# Subject-level table: one row per cohort subject; windows receive these values only through a many-to-one join.
static_features = bio.merge(gut, on="subject", how="outer", validate="one_to_one")
assert static_features["subject"].is_unique and set(static_features["subject"]) == set(windows["subject"])

# Range candidates and confirmed errors from the outlier review, mapped to static columns through the static-column traceability table.
trace = pd.read_csv(ALIGN_DIR / "static_column_traceability.csv")
trace = trace.loc[trace["table"].eq("bio")].assign(static_column=lambda d: "bio__" + d["canonical_column_name"].astype(str).str.strip())
outliers = pd.read_csv(ALIGN_DIR / "outlier_candidates.csv")
outliers = outliers.loc[outliers["data_source"].eq("bio")].merge(trace[["raw_column_name", "static_column"]],
                                                                  left_on="canonical_field", right_on="raw_column_name", how="left")

availability_by_feature = availability_registry.set_index("candidate_feature")
feature_rows = []
for column in static_features.columns.drop("subject"):
    source = source_of[column]
    if column.startswith("gut_"):
        role, group = "optional Gut supplement", "Gut"
        encoding = ("availability flag" if column == "gut_values_available" else
                    "ordinal 1-3 (1 = Not Optimal, 2 = Average, 3 = Good)" if column.startswith("gut_ord_") else
                    "categorical indicator for one level")
        registry_key = source.replace("gut__", "gut_health__") if source.startswith("gut__") else None
    else:
        role = "offline laboratory sensitivity" if column.startswith("bio_lab_") else "core"
        group = "B"
        encoding = ("availability flag" if column == "bio_values_available" else
                    f"indicator (reference level: {gender_reference})" if "gender" in column else "raw numeric value, no transformation")
        registry_key = source if source.startswith("bio__") else None
    entry = availability_by_feature.loc[registry_key] if registry_key in availability_by_feature.index else None
    subjects_with = static_features.loc[static_features[column].notna(), "subject"]
    windows_with = admitted_folds.loc[admitted_folds["subject"].isin(subjects_with)]
    raw_field = trace.set_index("static_column")["raw_column_name"].get(source)
    feature_rows.append({
        "record_type": "feature", "feature_name": column, "group_code": group, "role": role, "source_column": source,
        "encoding": encoding,
        "safe_for_prediction_at_t": entry["safe_for_prediction_at_t"] if entry is not None else "record flag",
        "availability_assumption": (entry["availability_time_definition"] if entry is not None else "record presence"),
        "n_subjects_with_value": int(len(subjects_with)), "n_cohort_subjects": len(static_features),
        **{f"subjects_protocol_b_{s}": int(len(set(subjects_with) & set(admitted_folds.loc[admitted_folds['evaluation_protocol'].eq(PROTOCOL_B) & admitted_folds['split'].eq(s), 'subject'])))
           for s in ["train", "validation", "test"]},
        "windows_covered_protocol_a": int(windows_with["evaluation_protocol"].eq(PROTOCOL_A).sum()),
        "windows_covered_protocol_b": int(windows_with["evaluation_protocol"].eq(PROTOCOL_B).sum()),
        "range_candidates": int(outliers["static_column"].eq(source).sum()),
        "confirmed_errors": int((outliers["static_column"].eq(source) & outliers["evidence_class"].str.startswith("confirmed")).sum()),
    })
excluded_rows = [{"record_type": "excluded", "feature_name": f.replace("bio__", "bio_lab_"), "group_code": "B",
                  "role": "excluded until unit confirmed", "source_column": f,
                  "encoding": "not produced",
"safe_for_prediction_at_t": availability_by_feature.loc[f, "safe_for_prediction_at_t"],
                  "availability_assumption": "HbA1c unit (percent vs mmol/mol) unresolved upstream; no confirmed assumption version"}
                 for f in BIO_PENDING]
static_feature_coverage = pd.DataFrame(feature_rows + excluded_rows)
assert static_feature_coverage.loc[static_feature_coverage["record_type"].eq("feature"), "confirmed_errors"].eq(0).all()

display(static_feature_coverage.loc[static_feature_coverage["group_code"].eq("B"),
                                    ["feature_name", "role", "encoding", "n_subjects_with_value", "range_candidates"]])

,feature_name,role,encoding,n_subjects_with_value,range_candidates
0,bio_age,core,"raw numeric value, no transformation",45.0,0.0
1,bio_gender_m,core,indicator (reference level: F),45.0,0.0
2,bio_bmi,core,"raw numeric value, no transformation",45.0,1.0
3,bio_lab_fasting_glucose_lab_mg_dl,offline laboratory sensitivity,"raw numeric value, no transformation",45.0,0.0
4,bio_lab_fasting_insulin_raw,offline laboratory sensitivity,"raw numeric value, no transformation",45.0,0.0
5,bio_lab_triglycerides_mg_dl,offline laboratory sensitivity,"raw numeric value, no transformation",45.0,0.0
6,bio_lab_hdl_mg_dl,offline laboratory sensitivity,"raw numeric value, no transformation",45.0,0.0
7,bio_values_available,core,availability flag,45.0,0.0
97,bio_lab_hba1c_lab_raw,excluded until unit confirmed,not produced,NaN,NaN


Coverage is reported by subject and joinable window. Gut comparisons use subjects with Gut data; the BMI range candidate remains unchanged.


In [17]:
# --- Validate static feature integration ---

audit = {}
joined = windows.merge(static_features, on="subject", how="left", validate="many_to_one")
audit["every window subject has a static row"] = joined["bio_values_available"].notna().all()

# --- Confirm source fields are used appropriately ---

# Only the defined core and sensitivity Bio fields.
used_sources = set(static_feature_coverage.loc[static_feature_coverage["record_type"].eq("feature") & static_feature_coverage["group_code"].eq("B"), "source_column"]) - {"bio record"}
audit["Bio sources limited to the core and offline laboratory fields"] = used_sources <= set(BIO_CORE + BIO_SENSITIVITY)

# --- Confirm feature availability classes ---

cov = static_feature_coverage.loc[static_feature_coverage["record_type"].eq("feature")]
audit["core Bio usable after baseline"] = bool(cov.loc[cov["role"].eq("core") & ~cov["feature_name"].eq("bio_values_available"), "safe_for_prediction_at_t"].eq("conditional_after_baseline_availability").all())
audit["offline laboratory fields flagged as result-timing conditional"] = bool(cov.loc[cov["role"].str.startswith("offline laboratory"), "safe_for_prediction_at_t"].eq("conditional_after_verified_result_availability").all())
audit["Gut fields flagged as offline-only"] = bool(cov.loc[cov["group_code"].eq("Gut") & ~cov["feature_name"].eq("gut_values_available"), "safe_for_prediction_at_t"].str.startswith("no_unless").all())
audit["gender reference from Protocol B training subjects"] = gender_reference in set(static.loc[static["subject"].isin(b_train_subjects), "bio__gender"])

audit_table = pd.Series(audit, name="passed").to_frame()
assert audit_table["passed"].all(), audit_table

In [18]:
# Persist static features and their coverage table.
static_features_path = ABLATION_DIR / "static_features.parquet"
static_coverage_path = ABLATION_DIR / "static_feature_coverage.csv"
static_features.to_parquet(static_features_path, index=False)
static_feature_coverage.to_csv(static_coverage_path, index=False, encoding="utf-8", lineterminator="\n")

`static_feature_coverage.csv` records source, role, encoding, availability and subject/window coverage. Bio and Gut inputs are restricted to the specified core and sensitivity fields.


## Common-cohort model comparison

Fit each manual-feature configuration on its common cohort. Sparacino is univariate and GlucoBench LR retains its glucose-lag OLS definition, so exogenous groups are compared using Ridge (primary) and shallow GBT (secondary). Activity requires an observed heart-rate or activity-calorie minute in the history; Diet and Bio are available throughout, and Gut requires participant-level measurements. Core comparisons and diet-delay/laboratory sensitivities use the core cohort; Gut encodings use a Gut-available cohort with a matched reference. The full-cohort variant handles missing inputs explicitly. Fit imputation/scaling on TRAIN and select hyperparameters by validation subject-mean RMSE; TEST labels and features do not enter fitting.


In [19]:
# --- Define model families and hyperparameter grids ---

FAMILIES = {"ridge": "ridge_glucose_history_features", "shallow_gbt": "shallow_gbt_glucose_history_features"}
GRIDS = {family: baseline_registry["tuning_grids"][model_id] for family, model_id in FAMILIES.items()}
HYPERPARAMETER_COLUMNS = ("alpha", "max_iter", "max_depth", "learning_rate")

activity_diet = features
static_coverage = static_feature_coverage

# --- Assemble feature lists from the registries ---

# Feature lists per group, taken from the registries.
features_by_group = {
    "G": feature_group_registry.loc[feature_group_registry["record_type"].eq("glucose_history_feature"), "candidate_feature"].tolist(),
    "A": activity_diet_feature_registry.loc[activity_diet_feature_registry["group_code"].eq("A"), "feature_name"].tolist(),
    "D": activity_diet_feature_registry.loc[activity_diet_feature_registry["group_code"].eq("D") & activity_diet_feature_registry["variant"].eq("primary"), "feature_name"].tolist(),
    "D_delayed": activity_diet_feature_registry.loc[activity_diet_feature_registry["variant"].str.startswith("delayed"), "feature_name"].tolist(),
    "B": static_coverage.loc[static_coverage["role"].eq("core") & ~static_coverage["encoding"].eq("availability flag"), "feature_name"].tolist(),
    "B_lab": static_coverage.loc[static_coverage["role"].str.startswith("offline laboratory") & static_coverage["record_type"].eq("feature"), "feature_name"].tolist(),
    "Gut_ordinal": static_coverage.loc[static_coverage["feature_name"].str.startswith("gut_ord_"), "feature_name"].tolist(),
    "Gut_categorical": static_coverage.loc[static_coverage["feature_name"].str.startswith("gut_cat_"), "feature_name"].tolist(),
}
display(pd.Series({g: len(f) for g, f in features_by_group.items()}, name="features").to_frame())

,features
G,10
A,26
D,21
D_delayed,21
B,3
B_lab,4
Gut_ordinal,22
Gut_categorical,66


In [20]:
# Window-level availability per group, computed from features only (no labels); membership for every admitted fold row.
window_availability = activity_diet[WINDOW_KEY + ["act_hr_missing_120m", "act_calories_missing_120m"]].merge(
    static_features[["subject", "bio_values_available", "gut_values_available"]], on="subject", how="left", validate="many_to_one")
window_availability["available_G"] = True
window_availability["available_A"] = window_availability["act_hr_missing_120m"].eq(0) | window_availability["act_calories_missing_120m"].eq(0)
window_availability["available_D"] = True   # unknown intake is represented by the diet flags, not by removing windows
window_availability["available_B"] = window_availability["bio_values_available"].eq(1)
window_availability["available_Gut"] = window_availability["gut_values_available"].eq(1)
cohort_rows = all_folds.loc[all_folds["admitted_to_split"]].drop(columns="admitted_to_split")
cohorts = cohort_rows.merge(window_availability[WINDOW_KEY + [c for c in window_availability.columns if c.startswith("available_")]],
                            on=WINDOW_KEY, how="left", validate="many_to_one")
cohorts["cohort_full"] = True
cohorts["cohort_core"] = cohorts[["available_G", "available_A", "available_D", "available_B"]].all(axis=1)
cohorts["cohort_gut"] = cohorts["cohort_core"] & cohorts["available_Gut"]
COHORTS = {"cohort_core": "core common cohort (G, A, D, B available)",
           "cohort_gut": "Gut common cohort (core cohort and Gut available)",
           "cohort_full": "all admitted windows (missing-handling version)"}

# Coverage and loss relative to all admitted windows, per protocol and split.
coverage_rows = []
for (protocol, split), part in cohorts.groupby(["evaluation_protocol", "split"]):
    for cohort, description in COHORTS.items():
        inside = part.loc[part[cohort]]
        coverage_rows.append({"evaluation_protocol": protocol, "split": split, "cohort": cohort, "description": description,
                              "windows": len(inside), "windows_admitted": len(part), "windows_lost": len(part) - len(inside),
                              "subjects": inside["subject"].nunique(), "subjects_admitted": part["subject"].nunique(),
                              "subjects_lost": part["subject"].nunique() - inside["subject"].nunique()})
cohort_coverage = pd.DataFrame(coverage_rows)
display(cohort_coverage.loc[cohort_coverage["split"].isin(["train", "validation"]),
                            ["evaluation_protocol", "split", "cohort", "windows", "windows_lost", "subjects", "subjects_lost"]])

,evaluation_protocol,split,cohort,windows,windows_lost,subjects,subjects_lost
3,Unseen-subject Generalization,train,cohort_core,79294,2664,27,0
4,Unseen-subject Generalization,train,cohort_gut,73153,8805,25,2
5,Unseen-subject Generalization,train,cohort_full,81958,0,27,0
6,Unseen-subject Generalization,validation,cohort_core,23139,1120,9,0
7,Unseen-subject Generalization,validation,cohort_gut,23139,1120,9,0
8,Unseen-subject Generalization,validation,cohort_full,24259,0,9,0
12,Within-subject Future Prediction,train,cohort_core,79300,0,45,0
13,Within-subject Future Prediction,train,cohort_gut,73134,6166,42,3
14,Within-subject Future Prediction,train,cohort_full,79300,0,45,0
15,Within-subject Future Prediction,validation,cohort_core,26301,29,45,0


The core cohort requires observed activity; the Gut cohort also requires Gut values. Coverage tables report the resulting exclusions by protocol and split.


In [21]:
# Fit plan: every configuration of the ablation matrix on its assigned cohort, for both protocols and both families.
configurations = ablation_matrix.loc[ablation_matrix["record_type"].eq("configuration"), ["config_id", "feature_groups"]]
plan_rows = [{"config_id": r.config_id, "groups": r.feature_groups.split("+"), "cohort": cohort, "variant": variant}
             for cohort, variant in [("cohort_core", "common cohort"), ("cohort_full", "missing-handling version")]
             for r in configurations.itertuples(index=False)]
plan_rows += [
    {"config_id": "diet_delay", "groups": ["G", "A", "D_delayed", "B"], "cohort": "cohort_core", "variant": "common cohort"},
    {"config_id": "offline_lab_bio", "groups": ["G", "A", "D", "B", "B_lab"], "cohort": "cohort_core", "variant": "common cohort"},
    {"config_id": "G+A+D+B", "groups": ["G", "A", "D", "B"], "cohort": "cohort_gut", "variant": "Gut reference"},
    {"config_id": "gut_ordinal", "groups": ["G", "A", "D", "B", "Gut_ordinal"], "cohort": "cohort_gut", "variant": "common cohort"},
    {"config_id": "gut_categorical", "groups": ["G", "A", "D", "B", "Gut_categorical"], "cohort": "cohort_gut", "variant": "common cohort"},
]
fit_plan = pd.DataFrame(plan_rows)
fit_plan["n_features"] = fit_plan["groups"].apply(lambda gs: sum(len(features_by_group[g]) for g in gs))

# Modeling table: admitted train/validation rows with labels; test rows are never joined to labels or features.
labels = samples[WINDOW_KEY + ["target_value"] + features_by_group["G"]].copy()
data = (cohorts.loc[cohorts["split"].isin(["train", "validation"])]
        .merge(labels, on=WINDOW_KEY, how="left", validate="many_to_one")
        .merge(activity_diet, on=WINDOW_KEY, how="left", validate="many_to_one")
        .merge(static_features, on="subject", how="left", validate="many_to_one"))
assert data["target_value"].notna().all() and set(data["split"]) == {"train", "validation"}

In [22]:
# --- Define model construction (error summaries come from src.evaluation) ---

def build_model(family: str, params: dict):
    if family == "ridge":
        # Imputation and scaling are fitted inside the pipeline on training rows only.
        return make_pipeline(SimpleImputer(strategy="median", keep_empty_features=True), StandardScaler(), Ridge(**params))
    return HistGradientBoostingRegressor(**params, early_stopping=False, random_state=0)

# --- Fit every planned configuration and collect validation predictions ---

fitted, fit_rows, validation_subject_rows = {}, [], {}
for row in fit_plan.itertuples(index=False):
    columns = [c for g in row.groups for c in features_by_group[g]]
    for protocol in (PROTOCOL_A, PROTOCOL_B):
        part = data.loc[data["evaluation_protocol"].eq(protocol) & data[row.cohort]]
        train, valid = part.loc[part["split"].eq("train")], part.loc[part["split"].eq("validation")]
        for family, grid in GRIDS.items():
            for params in grid:
                config = ",".join(f"{k}={v}" for k, v in params.items())
                start = time.perf_counter()
                model = build_model(family, params).fit(train[columns], train["target_value"])
                runtime = time.perf_counter() - start
                key = "|".join([row.config_id, row.cohort, protocol, family, config])
                fitted[key] = {"model": model, "inputs": columns}
                for split, frame in [("train", train), ("validation", valid)]:
                    prediction = model.predict(frame[columns])
                    metrics = evaluate(frame, prediction)
                    per_subject_rows = subject_metrics(frame, prediction).to_dict("records")
                    fit_rows.append({"fit_key": key, "config_id": row.config_id, "feature_groups": "+".join(row.groups),
                                     "cohort": row.cohort, "variant": row.variant, "evaluation_protocol": protocol,
                                     "family": family, "hyperparameters": config,
                                     **{name: params.get(name, np.nan) for name in HYPERPARAMETER_COLUMNS}, "n_features": len(columns),
                                     "split": split, **metrics,
                                     "fit_seconds": runtime if split == "train" else np.nan})
                    if split == "validation":
                        validation_subject_rows[key] = per_subject_rows
fit_table = pd.DataFrame(fit_rows)


In [23]:
# Hyperparameter selection per configuration, cohort, protocol and family: lowest validation subject-mean RMSE.
group_cols = ["config_id", "cohort", "evaluation_protocol", "family"]
validation_rows = fit_table.loc[fit_table["split"].eq("validation")]
selected = validation_rows.sort_values("subject_mean_rmse", kind="stable").groupby(group_cols, sort=False).head(1)
fit_table["selected"] = fit_table["fit_key"].isin(selected["fit_key"])

# Exact window-key equality, established once: every configuration assigned to a given (cohort, protocol, split)
# is fit and scored on the row set data.loc[...cohort & protocol & split...] used by the fit loop above, which
# does not depend on config_id, family or hyperparameters. Any two configurations sharing a cohort (including
# every planned contrast, sensitivity, Gut variant, and ridge/shallow_gbt structure pair below) are
# therefore paired on identical windows. This is checked against an independently derived key set from
# `cohorts` (the pre-merge source, built before the label/activity/static joins) with an outer merge and
# indicator=True: no key may be missing or duplicated on either side.
for (cohort, protocol, split), part in fit_table.groupby(["cohort", "evaluation_protocol", "split"]):
    modeling_keys = data.loc[data["evaluation_protocol"].eq(protocol) & data[cohort] & data["split"].eq(split), WINDOW_KEY]
    expected_keys = cohorts.loc[cohorts["evaluation_protocol"].eq(protocol) & cohorts[cohort] & cohorts["split"].eq(split), WINDOW_KEY]
    assert not expected_keys.empty
    assert not modeling_keys.duplicated().any() and not expected_keys.duplicated().any()
    key_merge = modeling_keys.merge(expected_keys, on=WINDOW_KEY, how="outer", indicator=True)
    assert (key_merge["_merge"] == "both").all(), key_merge["_merge"].value_counts()
    assert part["windows"].eq(len(expected_keys)).all()

# Per-subject validation metrics of the selected fits (inputs for later gain analysis; no gains computed here).
subject_rows = []
for s in selected.itertuples(index=False):
    for record in validation_subject_rows[s.fit_key]:
        subject_rows.append({"fit_key": s.fit_key, "config_id": s.config_id, "cohort": s.cohort, "evaluation_protocol": s.evaluation_protocol,
                             "family": s.family, "hyperparameters": s.hyperparameters, "split": "validation", "subject": record["subject"],
                             "mae": record["mae"], "rmse": record["rmse"], "windows": record["windows"]})
subject_table = pd.DataFrame(subject_rows)
display(selected.pivot_table(index=["cohort", "config_id"], columns=["evaluation_protocol", "family"], values="hyperparameters", aggfunc="first"))

evaluation_protocol         Unseen-subject Generalization                                             Within-subject Future Prediction  \
family                                              ridge                                 shallow_gbt                            ridge   
cohort      config_id                                                                                                                    
cohort_core G                                   alpha=0.1  learning_rate=0.1,max_depth=3,max_iter=300                        alpha=0.1   
            G+A                                 alpha=0.1  learning_rate=0.1,max_depth=3,max_iter=300                        alpha=0.1   
            G+A+B                               alpha=0.1  learning_rate=0.1,max_depth=3,max_iter=100                        alpha=0.1   
            G+A+D                               alpha=0.1  learning_rate=0.1,max_depth=3,max_iter=300                        alpha=0.1   
            G+A+D+B                             alpha=0.1  learning_rate=0.1,max_depth=3,max_iter=100                        alpha=0.1   
            G+B                                 alpha=0.1  learning_rate=0.1,max_depth=3,max_iter=100                        alpha=0.1   
            G+D                                 alpha=0.1  learning_rate=0.1,max_depth=3,max_iter=300                        alpha=0.1   
            G+D+B                               alpha=0.1  learning_rate=0.1,max_depth=3,max_iter=100                        alpha=0.1   
            diet_delay                          alpha=0.1  learning_rate=0.1,max_depth=3,max_iter=100                        alpha=0.1   
            offline_lab_bio                     alpha=0.1  learning_rate=0.1,max_depth=3,max_iter=100                        alpha=0.1   
cohort_full G                                   alpha=0.1  learning_rate=0.1,max_depth=3,max_iter=300                        alpha=0.1   
            G+A                                 alpha=0.1  learning_rate=0.1,max_depth=3,max_iter=300                        alpha=0.1   
            G+A+B                               alpha=0.1  learning_rate=0.1,max_depth=3,max_iter=300                        alpha=0.1   
            G+A+D                               alpha=0.1  learning_rate=0.1,max_depth=3,max_iter=300                        alpha=0.1   
            G+A+D+B                             alpha=0.1  learning_rate=0.1,max_depth=3,max_iter=100                        alpha=0.1   
            G+B                                 alpha=0.1  learning_rate=0.1,max_depth=3,max_iter=100                        alpha=0.1   
            G+D                                 alpha=0.1  learning_rate=0.1,max_depth=3,max_iter=300                        alpha=0.1   
            G+D+B                               alpha=0.1  learning_rate=0.1,max_depth=3,max_iter=100                        alpha=0.1   
cohort_gut  G+A+D+B                             alpha=0.1  learning_rate=0.1,max_depth=3,max_iter=100                        alpha=0.1   
            gut_categorical                     alpha=0.1  learning_rate=0.1,max_depth=3,max_iter=100                        alpha=0.1   
            gut_ordinal                        alpha=10.0  learning_rate=0.1,max_depth=3,max_iter=100                        alpha=0.1   

evaluation_protocol                                                      
family                                                      shallow_gbt  
cohort      config_id                                                    
cohort_core G                learning_rate=0.1,max_depth=3,max_iter=300  
            G+A              learning_rate=0.1,max_depth=3,max_iter=300  
            G+A+B            learning_rate=0.1,max_depth=3,max_iter=300  
            G+A+D            learning_rate=0.1,max_depth=3,max_iter=300  
            G+A+D+B          learning_rate=0.1,max_depth=3,max_iter=300  
            G+B              learning_rate=0.1,max_depth=3,max_iter=300  
            G+D      

Selected Ridge penalties often lie at the lower grid boundary (\(\alpha=0.1\)); the grid was not extended. GBT uses 100 or 300 iterations. Effects therefore reflect the evaluated tuning budgets, not an exhaustive model search.


In [24]:
# --- Verify model-input isolation ---

isolation = {}
isolation["no test row in fitted or scored data"] = data.merge(all_folds.loc[all_folds["split"].eq("test"), FOLD_KEY], on=FOLD_KEY).empty
isolation["Protocol B training subjects disjoint from its validation subjects"] = set(
    data.loc[data["evaluation_protocol"].eq(PROTOCOL_B) & data["split"].eq("train"), "subject"]).isdisjoint(
    data.loc[data["evaluation_protocol"].eq(PROTOCOL_B) & data["split"].eq("validation"), "subject"])

fit_metadata = fit_table.drop_duplicates("fit_key").set_index("fit_key")
isolation["the diet-delay sensitivity uses delayed diet features only"] = all(not (set(f["inputs"]) & set(features_by_group["D"])) for k, f in fitted.items() if fit_metadata.at[k, "config_id"] == "diet_delay")
isolation["Gut fits restricted to the Gut common cohort"] = fit_metadata.loc[fit_metadata["config_id"].isin(["gut_ordinal", "gut_categorical"]), "cohort"].eq("cohort_gut").all()
isolation_table = pd.Series(isolation, name="passed").to_frame()
assert isolation_table["passed"].all(), isolation_table

In [25]:
# Persist comparison cohorts and fitted-model summaries (ablation_fits.joblib holds only validation-selected fits).
cohorts.to_parquet(ABLATION_DIR / "comparison_cohorts.parquet", index=False)
cohort_coverage.to_csv(ABLATION_DIR / "comparison_cohorts_coverage.csv", index=False, encoding="utf-8", lineterminator="\n")
selected_fit_keys = set(selected["fit_key"])
joblib.dump({"families": FAMILIES, "grids": GRIDS, "features_by_group": features_by_group,
             "fits": {k: v for k, v in fitted.items() if k in selected_fit_keys}}, ABLATION_DIR / "ablation_fits.joblib")
ablation_fits = pd.concat([fit_table.assign(scope="summary"), subject_table.assign(scope="subject", selected=True)], ignore_index=True)
ablation_fits.to_csv(ABLATION_DIR / "ablation_fits.csv", index=False, encoding="utf-8", lineterminator="\n")

Both sides of each contrast use identical training and validation windows.


## Validation gains and negative results

Validation comparisons reuse the selected common-cohort fits without refitting or reading TEST. Pair per-subject MAE/RMSE differences (to minus from). The descriptive label is improved when \(\Delta\mathrm{RMSE}\le-0.1\) mg/dL and more than half the subjects improve, worse when \(\Delta\mathrm{RMSE}\ge0.1\) and fewer than half improve, negligible when \(\lvert\Delta\mathrm{RMSE}\rvert<0.1\), and mixed otherwise. Ridge core comparisons are primary; GBT, full-cohort and sensitivity results remain separate.


In [26]:
# Select validation fits and define comparison constants.
summary = selected.assign(scope="summary")
subjects = subject_table.assign(scope="subject", selected=True)

NEGLIGIBLE_MG_DL = 0.1
FAMILY_ORDER = list(FAMILIES)
PRIMARY_FAMILY = "ridge"
FULL_MODEL = "+".join(FULL)
print(f"Outcome rule: improved if subject-mean ΔRMSE <= -{NEGLIGIBLE_MG_DL} and >50% subjects improve; worse if ΔRMSE >= "
      f"+{NEGLIGIBLE_MG_DL} and <50% improve; negligible if |ΔRMSE| < {NEGLIGIBLE_MG_DL}; otherwise mixed")

Outcome rule: improved if subject-mean ΔRMSE <= -0.1 and >50% subjects improve; worse if ΔRMSE >= +0.1 and <50% improve; negligible if |ΔRMSE| < 0.1; otherwise mixed


In [27]:
# --- Define validation-contrast helpers ---

def selected_key(config_id, cohort, protocol, family):
    rows = summary.loc[summary["config_id"].eq(config_id) & summary["cohort"].eq(cohort)
                       & summary["evaluation_protocol"].eq(protocol) & summary["family"].eq(family) & summary["split"].eq("validation")]
    assert len(rows) == 1, (config_id, cohort, protocol, family)
    return rows.iloc[0]


# --- Assemble paired validation contrasts ---

contrast_rows, delta_rows = [], []


def paired(block, contrast_id, label, group_tested, cohort, protocol, from_row, to_row):
    """Subject-paired deltas (to minus from); negative values mean lower error in the 'to' model."""
    a = subjects.loc[subjects["fit_key"].eq(from_row["fit_key"]), ["subject", "mae", "rmse"]]
    b = subjects.loc[subjects["fit_key"].eq(to_row["fit_key"]), ["subject", "mae", "rmse"]]
    m = a.merge(b, on="subject", suffixes=("_from", "_to"), validate="one_to_one")
    assert len(m) == len(a) == len(b)  # identical subjects on both sides
    m["d_mae"], m["d_rmse"] = m["mae_to"] - m["mae_from"], m["rmse_to"] - m["rmse_from"]
    val_cov = cohort_coverage.loc[cohort_coverage["cohort"].eq(cohort) & cohort_coverage["evaluation_protocol"].eq(protocol)].set_index("split")
    improved_share = float((m["d_rmse"] < 0).mean())
    d_rmse = float(m["d_rmse"].mean())
    contrast_rows.append({
        "block": block, "contrast_id": contrast_id, "contrast": label, "group_tested": group_tested, "cohort": cohort,
        "evaluation_protocol": protocol, "family_from": from_row["family"], "family_to": to_row["family"],
        "config_from": from_row["config_id"], "config_to": to_row["config_id"],
        "subjects": len(m), "validation_windows": int(to_row["windows"]),
        "validation_windows_lost_by_cohort": int(val_cov.loc["validation", "windows_lost"]),
        "training_windows_lost_by_cohort": int(val_cov.loc["train", "windows_lost"]),
        "subjects_lost_by_cohort": int(val_cov.loc[["train", "validation"], "subjects_lost"].max()),
        "rmse_from": float(from_row["subject_mean_rmse"]), "rmse_to": float(to_row["subject_mean_rmse"]),
        "mean_d_mae": float(m["d_mae"].mean()), "mean_d_rmse": d_rmse,
        "median_d_mae": float(m["d_mae"].median()), "median_d_rmse": float(m["d_rmse"].median()),
        "pooled_d_mae": float(to_row["pooled_mae"] - from_row["pooled_mae"]),
        "pooled_d_rmse": float(to_row["pooled_rmse"] - from_row["pooled_rmse"]),
        "improved_subjects": int((m["d_rmse"] < 0).sum()), "improved_share": improved_share,
        "worsened_subjects": int((m["d_rmse"] > 0).sum()),
        "n_features_from": int(from_row["n_features"]), "n_features_to": int(to_row["n_features"]),
        "complexity_change": (f"{int(to_row['n_features']) - int(from_row['n_features']):+d} features"
                              if from_row["family"] == to_row["family"] else
                              f"{from_row['family']}({from_row['hyperparameters']}) -> {to_row['family']}({to_row['hyperparameters']})"),
        "hyperparameters_from": from_row["hyperparameters"], "hyperparameters_to": to_row["hyperparameters"],
        "outcome": classify_effect(d_rmse, improved_share, NEGLIGIBLE_MG_DL),
    })
    for r in m.itertuples(index=False):
        delta_rows.append({"block": block, "contrast_id": contrast_id, "evaluation_protocol": protocol,
                           "family": to_row["family"], "cohort": cohort, "subject": r.subject,
                           "d_mae": r.d_mae, "d_rmse": r.d_rmse, "rmse_from": r.rmse_from, "rmse_to": r.rmse_to})


# --- Evaluate all planned validation contrasts ---

core_contrasts = ablation_matrix.loc[ablation_matrix["record_type"].eq("contrast")]
for protocol in (PROTOCOL_A, PROTOCOL_B):
    for family in FAMILY_ORDER:
        # Core feature-group contrasts (same family, core common cohort) and the same contrasts on all windows.
        for block, cohort in [("core", "cohort_core"), ("missing_handling_all_windows", "cohort_full")]:
            for c in core_contrasts.itertuples(index=False):
                paired(block, c.config_id, f"{c.from_config} -> {c.to_config} ({c.contrast_type})", c.group_tested, cohort, protocol,
                       selected_key(c.from_config, cohort, protocol, family), selected_key(c.to_config, cohort, protocol, family))
        # Planned sensitivity contrasts, reported separately.
        base = selected_key(FULL_MODEL, "cohort_core", protocol, family)
        paired("sensitivity", "diet_delay", f"{FULL_MODEL} -> diet_delay (diet delayed availability)", "D delay", "cohort_core", protocol,
               base, selected_key("diet_delay", "cohort_core", protocol, family))
        paired("sensitivity", "offline_lab_bio", f"{FULL_MODEL} -> offline_lab_bio (+ offline laboratory Bio)", "B_lab", "cohort_core", protocol,
               base, selected_key("offline_lab_bio", "cohort_core", protocol, family))
        gut_base = selected_key(FULL_MODEL, "cohort_gut", protocol, family)
        for variant in ["gut_ordinal", "gut_categorical"]:
            paired("gut_optional", variant, f"{FULL_MODEL} -> {variant} (Gut cohort)", "Gut", "cohort_gut", protocol,
                   gut_base, selected_key(variant, "cohort_gut", protocol, family))
        paired("gut_optional", "gut_encoding", "gut_ordinal -> gut_categorical (encoding)", "Gut encoding", "cohort_gut", protocol,
               selected_key("gut_ordinal", "cohort_gut", protocol, family), selected_key("gut_categorical", "cohort_gut", protocol, family))
    for config_id in ablation_matrix.loc[ablation_matrix["record_type"].eq("configuration"), "config_id"]:
        paired("structure_x_feature_group", f"structure_{config_id}",
 f"ridge -> shallow_gbt ({config_id})", "model structure",
               "cohort_core", protocol, selected_key(config_id, "cohort_core", protocol, "ridge"),
               selected_key(config_id, "cohort_core", protocol, "shallow_gbt"))

contrast_table = pd.DataFrame(contrast_rows)
subject_deltas = pd.DataFrame(delta_rows)


In [28]:
# Core feature-group results, primary family (Ridge), with the secondary family alongside.
columns = ["contrast_id", "contrast", "group_tested", "mean_d_mae", "mean_d_rmse", "median_d_rmse", "pooled_d_rmse",
           "improved_subjects", "subjects", "improved_share", "complexity_change", "outcome"]

# Primary family (Ridge) across both protocols in one compact table; the secondary family (shallow GBT)
# and the full family x protocol breakdown remain available in the persisted validation_ablation_results.csv.
primary_part = contrast_table.loc[contrast_table["block"].eq("core") & contrast_table["family_to"].eq(PRIMARY_FAMILY)]
display(primary_part[["evaluation_protocol"] + columns].round(3))

# Largest per-subject worsening in each core contrast (primary family), with that subject's validation window count.
windows_per_subject = subjects.drop_duplicates(["evaluation_protocol", "cohort", "subject"])[["evaluation_protocol", "cohort", "subject", "windows"]]
worst = (subject_deltas.loc[subject_deltas["block"].eq("core") & subject_deltas["family"].eq(PRIMARY_FAMILY)]
         .sort_values("d_rmse", ascending=False).groupby(["evaluation_protocol", "contrast_id"]).head(1)
         .merge(windows_per_subject, on=["evaluation_protocol", "cohort", "subject"], how="left")
         .sort_values(["evaluation_protocol", "contrast_id"]))
display(worst[["evaluation_protocol", "contrast_id", "subject", "windows", "rmse_from", "rmse_to", "d_rmse"]].round(3))

,evaluation_protocol,contrast_id,contrast,group_tested,mean_d_mae,mean_d_rmse,median_d_rmse,pooled_d_rmse,improved_subjects,subjects,improved_share,complexity_change,outcome
0,Within-subject Future Prediction,G->G+A,G -> G+A (incremental; single_group_addition),A,0.095,0.078,-0.062,-0.070,35,45,0.778,+26 features,negligible
1,Within-subject Future Prediction,G+A->G+A+D,G+A -> G+A+D (incremental),D,-0.110,-0.122,-0.103,-0.113,26,45,0.578,+21 features,improved
2,Within-subject Future Prediction,G+A+D->G+A+D+B,G+A+D -> G+A+D+B (incremental; leave_one_group_out),B,0.004,-0.020,-0.053,-0.015,32,45,0.711,+3 features,negligible
3,Within-subject Future Prediction,G->G+D,G -> G+D (single_group_addition),D,-0.118,-0.133,-0.088,-0.143,26,45,0.578,+21 features,improved
4,Within-subject Future Prediction,G->G+B,G -> G+B (single_group_addition),B,-0.016,-0.040,-0.031,-0.012,32,45,0.711,+3 features,negligible
5,Within-subject Future Prediction,G+D+B->G+A+D+B,G+D+B -> G+A+D+B (leave_one_group_out),A,0.128,0.113,-0.041,-0.045,33,45,0.733,+26 features,mixed
6,Within-subject Future Prediction,G+A+B->G+A+D+B,G+A+B -> G+A+D+B (leave_one_group_out),D,-0.108,-0.124,-0.099,-0.112,27,45,0.600,+21 features,improved
46,Unseen-subject Generalization,G->G+A,G -> G+A (incremental; single_group_addition),A,-0.010,0.072,-0.034,-0.004,5,9,0.556,+26 features,negligible
47,Unseen-subject Generalization,G+A->G+A+D,G+A -> G+A+D (incremental),D,-0.039,-0.094,-0.053,-0.136,7,9,0.778,+21 features,negligible
48,Unseen-subject Generalization,G+A+D->G+A+D+B,G+A+D -> G+A+D+B (incremental; leave_one_group_out),B,0.021,0.016,0.005,0.006,4,9,0.444,+3 features,negligible


,evaluation_protocol,contrast_id,subject,windows,rmse_from,rmse_to,d_rmse
10,Unseen-subject Generalization,G+A+B->G+A+D+B,7,110,15.437,15.648,0.211
11,Unseen-subject Generalization,G+A+D->G+A+D+B,7,110,15.437,15.648,0.210
12,Unseen-subject Generalization,G+A->G+A+D,7,110,15.262,15.437,0.175
2,Unseen-subject Generalization,G+D+B->G+A+D+B,7,110,14.556,15.648,1.092
4,Unseen-subject Generalization,G->G+A,7,110,14.546,15.262,0.717
9,Unseen-subject Generalization,G->G+B,7,110,14.546,14.785,0.239
13,Unseen-subject Generalization,G->G+D,18,2755,9.798,9.852,0.053
6,Within-subject Future Prediction,G+A+B->G+A+D+B,30,589,17.089,17.790,0.702
8,Within-subject Future Prediction,G+A+D->G+A+D+B,35,546,12.530,13.054,0.524
5,Within-subject Future Prediction,G+A->G+A+D,30,589,17.233,17.936,0.703


Subject-equal and pooled-window effects can differ when subjects contribute unequal numbers of windows. The paired table retains both summaries.


In [29]:
# Separately reported blocks: sensitivity, optional Gut, missing-handling version, and model structure.
columns = ["block", "contrast_id", "contrast", "evaluation_protocol", "family_to", "mean_d_rmse", "improved_subjects",
           "subjects", "subjects_lost_by_cohort", "validation_windows_lost_by_cohort", "outcome"]
for block in ["sensitivity", "gut_optional", "structure_x_feature_group"]:
    part = contrast_table.loc[contrast_table["block"].eq(block)]
    part = part.loc[part["family_to"].eq(PRIMARY_FAMILY)] if block != "structure_x_feature_group" else part
    display(part[columns].round(3))
agree = (contrast_table.loc[contrast_table["block"].eq("core")].set_index(["contrast_id", "evaluation_protocol", "family_to"])["outcome"]
         .eq(contrast_table.loc[contrast_table["block"].eq("missing_handling_all_windows")].set_index(["contrast_id", "evaluation_protocol", "family_to"])["outcome"]))
print(f"Core and all-window (missing-handling) versions give the same outcome label in {int(agree.sum())} of {len(agree)} contrasts")

,block,contrast_id,contrast,evaluation_protocol,family_to,mean_d_rmse,improved_subjects,subjects,subjects_lost_by_cohort,validation_windows_lost_by_cohort,outcome
14,sensitivity,diet_delay,G+A+D+B -> diet_delay (diet delayed availability),Within-subject Future Prediction,ridge,0.173,17,45,0,29,worse
15,sensitivity,offline_lab_bio,G+A+D+B -> offline_lab_bio (+ offline laboratory Bio),Within-subject Future Prediction,ridge,-0.174,38,45,0,29,improved
60,sensitivity,diet_delay,G+A+D+B -> diet_delay (diet delayed availability),Unseen-subject Generalization,ridge,0.098,2,9,0,1120,negligible
61,sensitivity,offline_lab_bio,G+A+D+B -> offline_lab_bio (+ offline laboratory Bio),Unseen-subject Generalization,ridge,0.062,6,9,0,1120,negligible


,block,contrast_id,contrast,evaluation_protocol,family_to,mean_d_rmse,improved_subjects,subjects,subjects_lost_by_cohort,validation_windows_lost_by_cohort,outcome
16,gut_optional,gut_ordinal,G+A+D+B -> gut_ordinal (Gut cohort),Within-subject Future Prediction,ridge,-0.203,36,42,3,2035,improved
17,gut_optional,gut_categorical,G+A+D+B -> gut_categorical (Gut cohort),Within-subject Future Prediction,ridge,-0.346,40,42,3,2035,improved
18,gut_optional,gut_encoding,gut_ordinal -> gut_categorical (encoding),Within-subject Future Prediction,ridge,-0.144,34,42,3,2035,improved
62,gut_optional,gut_ordinal,G+A+D+B -> gut_ordinal (Gut cohort),Unseen-subject Generalization,ridge,3.483,2,9,2,1120,worse
63,gut_optional,gut_categorical,G+A+D+B -> gut_categorical (Gut cohort),Unseen-subject Generalization,ridge,0.992,4,9,2,1120,worse
64,gut_optional,gut_encoding,gut_ordinal -> gut_categorical (encoding),Unseen-subject Generalization,ridge,-2.491,7,9,2,1120,improved


,block,contrast_id,contrast,evaluation_protocol,family_to,mean_d_rmse,improved_subjects,subjects,subjects_lost_by_cohort,validation_windows_lost_by_cohort,outcome
38,structure_x_feature_group,structure_G,ridge -> shallow_gbt (G),Within-subject Future Prediction,shallow_gbt,1.497,7,45,0,29,worse
39,structure_x_feature_group,structure_G+A,ridge -> shallow_gbt (G+A),Within-subject Future Prediction,shallow_gbt,1.414,6,45,0,29,worse
40,structure_x_feature_group,structure_G+A+D,ridge -> shallow_gbt (G+A+D),Within-subject Future Prediction,shallow_gbt,1.148,8,45,0,29,worse
41,structure_x_feature_group,structure_G+A+D+B,ridge -> shallow_gbt (G+A+D+B),Within-subject Future Prediction,shallow_gbt,0.744,11,45,0,29,worse
42,structure_x_feature_group,structure_G+D,ridge -> shallow_gbt (G+D),Within-subject Future Prediction,shallow_gbt,1.191,7,45,0,29,worse
43,structure_x_feature_group,structure_G+B,ridge -> shallow_gbt (G+B),Within-subject Future Prediction,shallow_gbt,1.151,9,45,0,29,worse
44,structure_x_feature_group,structure_G+D+B,ridge -> shallow_gbt (G+D+B),Within-subject Future Prediction,shallow_gbt,0.746,11,45,0,29,worse
45,structure_x_feature_group,structure_G+A+B,ridge -> shallow_gbt (G+A+B),Within-subject Future Prediction,shallow_gbt,1.048,9,45,0,29,worse
84,structure_x_feature_group,structure_G,ridge -> shallow_gbt (G),Unseen-subject Generalization,shallow_gbt,1.986,1,9,0,1120,worse
85,structure_x_feature_group,structure_G+A,ridge -> shallow_gbt (G+A),Unseen-subject Generalization,shallow_gbt,2.328,1,9,0,1120,worse


Core and all-window (missing-handling) versions give the same outcome label in 25 of 28 contrasts


Gut, laboratory and model-family contrasts are reported separately from the core feature-group comparisons.


In [30]:
# Validation ablation figure: per-subject ΔRMSE for the core contrasts (primary family), both protocols.
fig, axes = plt.subplots(1, 2, figsize=(12, 4.8), sharey=True)
order = core_contrasts["config_id"].tolist()
labels = [f"{c.from_config}->{c.to_config}\n(+{c.group_tested})" for c in core_contrasts.itertuples(index=False)]
for ax, protocol in zip(axes, (PROTOCOL_A, PROTOCOL_B)):
    part = subject_deltas.loc[subject_deltas["block"].eq("core") & subject_deltas["family"].eq(PRIMARY_FAMILY)
                              & subject_deltas["evaluation_protocol"].eq(protocol)]
    for i, contrast_id in enumerate(order):
        values = part.loc[part["contrast_id"].eq(contrast_id), "d_rmse"].to_numpy()
        jitter = np.linspace(-0.15, 0.15, len(values)) if len(values) > 1 else np.zeros(1)
        ax.scatter(np.full(len(values), i) + jitter, values, s=10, alpha=0.5, color="tab:blue")
        ax.scatter([i], [values.mean()], s=60, marker="D", color="black", zorder=3)
    ax.axhline(0, color="grey", linewidth=0.8)
    ax.set_xticks(range(len(order)), labels, fontsize=7)
    ax.set_title(f"{protocol} (Ridge, core cohort)", fontsize=10)
axes[0].set_ylabel("Per-subject validation ΔRMSE (mg/dL); < 0 = lower error")
fig.suptitle("Validation ablation: per-subject paired change in RMSE (diamonds = subject mean)", fontsize=11)
fig.tight_layout()
figure_path = ABLATION_DIR / "validation_ablation.png"
fig.savefig(figure_path, dpi=120, metadata={"Software": None})
plt.close(fig)

In [31]:
# Validate paired contrast coverage and configuration identity.
isolation = {
    "core block complete (7 contrasts x 2 protocols x 2 families)": int(contrast_table["block"].eq("core").sum()) == len(core_contrasts) * 2 * len(FAMILIES),
    "subject deltas complete for every contrast": bool((subject_deltas.groupby(["block", "contrast_id", "evaluation_protocol", "family"]).size()
                                                        .values == contrast_table.set_index(["block", "contrast_id", "evaluation_protocol", "family_to"])
                                                        .loc[subject_deltas.groupby(["block", "contrast_id", "evaluation_protocol", "family"]).size().index, "subjects"].values).all()),
    "same family within feature-group blocks": bool(contrast_table.loc[contrast_table["block"].ne("structure_x_feature_group")]
                                                    .eval("family_from == family_to").all()),
    "structure contrasts on identical feature sets": bool(contrast_table.loc[contrast_table["block"].eq("structure_x_feature_group")]
                                                          .eval("config_from == config_to").all()),
}
assert all(isolation.values())

results_path = ABLATION_DIR / "validation_ablation_results.csv"
validation_ablation_results = pd.concat([contrast_table.assign(record_type="contrast"),
                                         subject_deltas.assign(record_type="subject_delta")], ignore_index=True)
validation_ablation_results.to_csv(results_path, index=False, encoding="utf-8", lineterminator="\n")
display(contrast_table.groupby(["block", "outcome"]).size().unstack(fill_value=0))

outcome,improved,mixed,negligible,worse
block,,,,
core,12,3,10,3
gut_optional,7,0,1,4
missing_handling_all_windows,14,2,9,3
sensitivity,2,1,2,3
structure_x_feature_group,0,0,0,16


Coverage tables quantify exclusions for each cohort and split; comparisons use matched windows.


## Manual ablation evaluation plan


Retain all planned Ridge configurations and sensitivities regardless of validation gains; GBT supplies the glucose-reference and full-model structure checks. TEST uses training-only fits without TRAIN+VALIDATION refitting. `evaluation_plan.json` records comparisons, bootstrap settings and cohort rules.


In [32]:
# Preserve protocol order when assigning experiment IDs.
PROTOCOLS = [PROTOCOL_A, PROTOCOL_B]

In [33]:
# Retain every Ridge configuration and sensitivity, regardless of validation gains.
# GBT contributes reference and full-model comparisons only.
# as the structure check. Hyperparameters are the validation selections of the model comparison; nothing is added or dropped after the validation gains.
TIERS = [
    ("primary", "ridge", "cohort_core", ablation_matrix.loc[ablation_matrix["record_type"].eq("configuration"), "config_id"].tolist()),
    ("sensitivity", "ridge", "cohort_core", ["diet_delay", "offline_lab_bio"]),
    ("gut_optional", "ridge", "cohort_gut", ["G+A+D+B", "gut_ordinal", "gut_categorical"]),
    ("missing_handling_all_windows", "ridge", "cohort_full", ablation_matrix.loc[ablation_matrix["record_type"].eq("configuration"), "config_id"].tolist()),
    ("structure_check", "shallow_gbt", "cohort_core", ["G", "G+A+D+B"]),
]
rows = []
for tier, family, cohort, config_ids in TIERS:
    for config_id in config_ids:
        for protocol in PROTOCOLS:
            match = selected.loc[selected["config_id"].eq(config_id) & selected["cohort"].eq(cohort)
                                 & selected["evaluation_protocol"].eq(protocol) & selected["family"].eq(family)]
            assert len(match) == 1, (tier, config_id, cohort, protocol, family)
            m = match.iloc[0]
            inputs = fitted[m["fit_key"]]["inputs"]
            rows.append({
                "experiment_id": f"{tier}|{config_id}|{cohort}|{family}|{protocol}", "tier": tier, "config_id": config_id,
                "feature_groups": m["feature_groups"], "n_features": len(inputs), "cohort": cohort,
                "evaluation_protocol": protocol, "family": family, "hyperparameters": m["hyperparameters"],
                "fit_key": m["fit_key"],
                "validation_subject_mean_rmse": float(m["subject_mean_rmse"]),
                "validation_subject_mean_mae": float(m["subject_mean_mae"]),
            })
planned_experiments = pd.DataFrame(rows)
assert planned_experiments["experiment_id"].is_unique and planned_experiments["fit_key"].is_unique
display(planned_experiments.groupby(["tier", "family", "cohort"]).agg(configurations=("config_id", "nunique"), experiments=("experiment_id", "size")))

,,,configurations,experiments
tier,family,cohort,,
gut_optional,ridge,cohort_gut,3,6
missing_handling_all_windows,ridge,cohort_full,8,16
primary,ridge,cohort_core,8,16
sensitivity,ridge,cohort_core,2,4
structure_check,shallow_gbt,cohort_core,2,4


In [34]:
# Select the validation contrasts whose fitted configurations enter test evaluation.
fixed_keys = set(zip(planned_experiments["config_id"], planned_experiments["cohort"], planned_experiments["evaluation_protocol"], planned_experiments["family"]))
contrast_all = validation_ablation_results.loc[validation_ablation_results["record_type"].eq("contrast")]
both_planned = contrast_all.apply(lambda c: (c["config_from"], c["cohort"], c["evaluation_protocol"], c["family_from"]) in fixed_keys
                                 and (c["config_to"], c["cohort"], c["evaluation_protocol"], c["family_to"]) in fixed_keys, axis=1)
contrast_specs = (contrast_all.loc[both_planned, ["block", "contrast_id", "contrast", "group_tested", "cohort", "evaluation_protocol",
                                                  "family_from", "family_to", "config_from", "config_to", "outcome"]]
                  .rename(columns={"outcome": "validation_outcome"}).reset_index(drop=True))

# Test sample protocol: cohort membership follows the feature-availability rule of the model comparison; counts are keys only.
test_coverage = cohort_coverage.loc[cohort_coverage["split"].eq("test"),
                                    ["evaluation_protocol", "cohort", "windows", "windows_admitted", "windows_lost", "subjects", "subjects_lost"]]

evaluation = {
    "uncertainty": {"method": "subject-level paired cluster bootstrap (all windows of a subject resampled together)",
                    "resamples": 2000, "seed": 42, "interval_level": 0.95, "interval_type": "percentile"},
    "outcome_rule": {"negligible_mg_dl": NEGLIGIBLE_MG_DL},
}
display(test_coverage)

,evaluation_protocol,cohort,windows,windows_admitted,windows_lost,subjects,subjects_lost
0,Unseen-subject Generalization,cohort_core,26861,28322,1461,9,0
1,Unseen-subject Generalization,cohort_gut,23455,28322,4867,8,1
2,Unseen-subject Generalization,cohort_full,28322,28322,0,9,0
9,Within-subject Future Prediction,cohort_core,21083,26269,5186,44,1
10,Within-subject Future Prediction,cohort_gut,19888,26269,6381,41,4
11,Within-subject Future Prediction,cohort_full,26269,26269,0,45,0


TEST cohort exclusions are reported in the coverage table. Baseline comparisons use identical windows.


In [35]:
# --- Serialize the planned experiment specification ---

evaluation_plan = {
    "feature_groups": {"G": "glucose-history features", "A": "activity features", "D": "primary diet features",
                       "D_delayed": "diet features (log time + 30 min)", "B": "core Bio",
                       "B_lab": "offline laboratory Bio", "Gut": "Gut ordinal or categorical"},
    "evaluation": evaluation,
    "test_sample": test_coverage.to_dict(orient="records"),
    "experiments": planned_experiments.to_dict(orient="records"),
    "contrasts": contrast_specs.to_dict(orient="records"),
}
(ABLATION_DIR / "evaluation_plan.json").write_text(json.dumps(evaluation_plan, indent=2, default=str) + "\n", encoding="utf-8")



No train+validation refit is used. Evaluation uses subject-mean MAE and RMSE (primary) and pooled metrics (secondary), paired per-subject deltas, and a subject-level paired cluster bootstrap (2,000 resamples, seed 42). `evaluation_plan.json` (46 experiments, 42 contrasts) is saved in `results/manual_ablation/`; the plan uses membership counts, without test labels or metrics.